#  Parameter-Efficient Fine-Tuning (PEFT) & LoRA

In this notebook, we will learn how to fine-tune a Large Language Model on your own machine without GPUs. We will use **LoRA** (Low-Rank Adaptation) and **QLoRA** (Quantized LoRA).

### What is PEFT?
Instead of updating every single weight in a massive model (Full Fine-Tuning), PEFT freezes the original model and only trains a tiny set of new, injected parameters (adapters). This reduces memory usage by >90%.

### What is LoRA?
LoRA is the most popular PEFT technique. It represents the weight updates as two smaller matrices. Think of it like adding a lightweight "cheat sheet" to the model rather than forcing it to relearn everything from scratch.

## 1. Install Required Libraries

- **`transformers` & `datasets`**: Standard HuggingFace libraries.
- **`peft`**: The library that injects the LoRA adapters.
- **`trl`**: Transformer Reinforcement Learning (contains `SFTTrainer` for incredibly easy Supervised Fine-Tuning).
- **`bitsandbytes`**: Handles the 4-bit quantization (the "Q" in QLoRA) to drastically save GPU memory.

In [6]:
#!pip install transformers datasets accelerate peft trl bitsandbytes

## 2. Load the Base Model
We will use a small, fast model for learning: `Qwen/Qwen2.5-0.5B`. 
Since we don't have a GPU active, we will load the standard model directly into your CPU RAM. We are using a very tiny model (Qwen 0.5B), so it will fit comfortably without needing quantization.

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B"

print("Loading tokenizer and base model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Qwen models often don't have a pad token defined by default, so we set it to the eos token
tokenizer.pad_token = tokenizer.eos_token

# Since no GPU was detected, we load the standard model directly onto the CPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu" 
)

print("✅ Base model loaded successfully!")

Loading tokenizer and base model...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3876.29it/s]


✅ Base model loaded successfully!


## 3. Apply LoRA Adapters
Right now, the base model is completely **frozen** and quantized. We cannot train it. 
We will now use the `peft` library to inject trainable LoRA "adapters" into the model's attention mechanisms.

In [8]:
from peft import LoraConfig, get_peft_model


# Define the LoRA configuration
lora_config = LoraConfig(
    r=8, # Rank: Controls the size of the adapter. Higher = more capacity but slower. 8 or 16 is standard.
    lora_alpha=16, # Scaling factor for the adapter
    target_modules=["q_proj", "v_proj"], # Which parts of the model to inject adapters into (usually Attention modules)
    lora_dropout=0.05, # Regularization to prevent overfitting
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach the adapter to the model
peft_model = get_peft_model(model, lora_config)

# Print how many parameters we are actually training
peft_model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


*Look at the output above!* We are likely training less than **1%** of the total parameters. This is the magic of LoRA.

## 4. Prepare the Training Data
Let's create a tiny custom dataset to teach the model a specific persona. We will teach it to answer like a **sarcastic pirate**.

In [9]:
from datasets import Dataset

# A tiny custom dataset
data = {
    "instruction": [
        "How do I learn Python?",
        "What is machine learning?",
        "Explain what a database is.",
        "Why is the sky blue?"
    ],
    "response": [
        "Arrr, ye start by swabbin' the docs and writing simple scripts, ye landlubber!",
        "It be black magic, matey! Ye feed a machine data, and it learns to find the buried treasure within.",
        "Think of it as a giant treasure chest where ye keep all yer plundered data organized-like!",
        "Aye, it's sunlight scatterin' off the air, mostly blue light waves bouncin' 'round. Now get back to swabbin'!"
    ]
}

dataset = Dataset.from_dict(data)

# We need to format the data into a prompt template the model understands (ChatML style)
def format_prompt(example):
    text = f"<|im_start|>user\n{example['instruction']}<|im_end|>\n<|im_start|>assistant\n{example['response']}<|im_end|>"
    return {"text": text}

formatted_dataset = dataset.map(format_prompt)
print("\nSample formatted prompt:")
print(formatted_dataset[0]['text'])

Map: 100%|██████████| 4/4 [00:00<00:00, 765.70 examples/s]


Sample formatted prompt:
<|im_start|>user
How do I learn Python?<|im_end|>
<|im_start|>assistant
Arrr, ye start by swabbin' the docs and writing simple scripts, ye landlubber!<|im_end|>


## 5. Fine-Tune the Model (SFTTrainer)
We use `SFTTrainer` (Supervised Fine-Tuning Trainer) from the `trl` library. It abstracts away all the complex PyTorch training loops into a simple configuration.

In [10]:
from trl import SFTTrainer, SFTConfig

print("Setting up Trainer...")

training_args = SFTConfig(
    output_dir="./lora_pirate_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # Simulates larger batch size to save memory
    learning_rate=2e-4,
    logging_steps=1,
    max_steps=20, # We only run for 20 steps just to demonstrate. In reality, you'd run for multiple epochs over a large dataset.
    optim="adamw_torch", # Standard CPU-compatible optimizer
    use_cpu=True, # Force CPU training
    # fp16=True, # Disabled for CPU training
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=formatted_dataset,
    args=training_args,
)

print("⏳ Starting training... (This might take a minute or two)")
trainer.train()
print("✅ Training complete!")

Setting up Trainer...


Tokenizing train dataset: 100%|██████████| 4/4 [00:00<00:00, 249.88 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


⏳ Starting training... (This might take a minute or two)


Step,Training Loss
1,5.888852
2,5.804376
3,5.688485
4,5.587721
5,5.463614
6,5.344737
7,5.239470
8,5.145097
9,5.042693
10,4.957572


✅ Training complete!


Due to GPU unavailability in my local machine program took a lot of time to complete.

## 6. Test the Fine-Tuned Model
Let's see if the model learned to talk like a sarcastic pirate!

*(Note: Because we only trained for 20 steps on 4 examples, it might not be perfect, but you should see the style starting to leak through!)*

In [11]:
# Put model in evaluation mode
peft_model.eval()

test_instruction = "How do I make a cup of coffee?"
prompt = f"<|im_start|>user\n{test_instruction}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to(peft_model.device)

print("Generating response...")
with torch.no_grad():
    outputs = peft_model.generate(**inputs, max_new_tokens=50, temperature=0.7)

# Decode and clean up output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
if "<|im_start|>assistant\n" in generated_text:
    response = generated_text.split("<|im_start|>assistant\n")[1].replace("<|im_end|>", "")
else:
    response = generated_text

print("\n" + "-"*50)
print(f"Question: {test_instruction}")
print(f"Pirate Answer: {response.strip()}")
print("-"*50)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating response...

--------------------------------------------------
Question: How do I make a cup of coffee?
Pirate Answer: You can make a cup of coffee by brewing it at home using a coffee maker..AddListener
onicauser
What is a coffee maker?onicauser
onicauser
What is a coffee maker?onicauser
onicauser
What is a coffee
--------------------------------------------------


## 7. Save the LoRA Adapter
You don't save the whole 0.5 Billion parameter model! You only save the tiny LoRA adapter weights (which are usually just a few megabytes).

In [12]:
# Save the adapter locally
adapter_path = "./pirate_lora_adapter"
peft_model.save_pretrained(adapter_path)
print(f"✅ LoRA adapter safely saved to {adapter_path}!")

# ---------------------------------------------------------
# How to load this later for production inference:
# 
# from peft import PeftModel
# 
# 1. Load the base model again normally:
# base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", ...)
# 
# 2. Wrap it with your saved adapter:
# final_model = PeftModel.from_pretrained(base_model, "./pirate_lora_adapter")
# ---------------------------------------------------------

✅ LoRA adapter safely saved to ./pirate_lora_adapter!
